# Gradient Boosting

**DS4DH · Module 07 — Machine Learning and Interpretability**

*Technique:* Gradient boosted trees, and the trade of interpretability for accuracy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/07b_gradient_boosting.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Module 05 used regression, which gives one interpretable number per predictor and
assumes the relationship is a straight line.

Gradient boosting drops that assumption. It fits a small tree, looks at what it
got wrong, fits another tree to those errors, and repeats — hundreds of times.
The result usually predicts better and can no longer be summarised by a
coefficient.

That is a real trade, and it should be made deliberately.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
TARGET = 'Total'
PREDICTORS = ['renter_owner_gap', 'tot_income', 'log_pop']
RANDOM_STATE = 42

X = feat[PREDICTORS]
y = feat[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)

linear = LinearRegression().fit(X_train, y_train)
gbm = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3,
    random_state=RANDOM_STATE).fit(X_train, y_train)

print(f'{"model":<22}{"test R²":>10}{"test MAE":>11}')
print('-' * 43)
for name, m in [('linear regression', linear), ('gradient boosting', gbm)]:
    p = m.predict(X_test)
    print(f'{name:<22}{r2_score(y_test, p):>10.3f}{mean_absolute_error(y_test, p):>11.3f}')

## Where the extra accuracy comes from

Boosting wins where the relationship bends. Plotting predicted STIR against
income for both models shows the linear model committed to a straight line and the
boosted model did not.

In [ ]:
grid = pd.DataFrame({
    'renter_owner_gap': feat['renter_owner_gap'].median(),
    'tot_income': np.linspace(feat['tot_income'].quantile(0.02),
                              feat['tot_income'].quantile(0.98), 120),
    'log_pop': feat['log_pop'].median(),
})[PREDICTORS]

fig, ax = plt.subplots()
ax.scatter(feat['tot_income'], feat[TARGET], s=18, alpha=0.35, label='CSDs')
ax.plot(grid['tot_income'], linear.predict(grid), lw=2, label='linear')
ax.plot(grid['tot_income'], gbm.predict(grid), lw=2, color='#E8663D', label='gradient boosting')
ax.set_xlabel('total household income ($)')
ax.set_ylabel('Total STIR (%)')
ax.set_title('Same data, two assumptions about shape')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Boosting can overfit too. Watch test error as trees are added.
stages = list(range(1, 301))
test_err = [mean_absolute_error(y_test, p) for p in gbm.staged_predict(X_test)]
train_err = [mean_absolute_error(y_train, p) for p in gbm.staged_predict(X_train)]

fig, ax = plt.subplots()
ax.plot(stages, train_err, label='train MAE')
ax.plot(stages, test_err, color='#E8663D', label='test MAE')
best_n = int(np.argmin(test_err)) + 1
ax.axvline(best_n, color='#888', ls=':', label=f'best n_estimators ≈ {best_n}')
ax.set_xlabel('number of trees')
ax.set_ylabel('MAE (percentage points)')
ax.set_title('More trees is not monotonically better')
ax.legend()
plt.tight_layout()
plt.show()

print(f'test MAE bottoms out around {best_n} trees at {min(test_err):.3f} pp')

### 🔧 Your turn 1

Change `learning_rate` to `0.3` and re-run.

A larger learning rate makes each tree count for more. Does the model reach its
best test error in fewer trees? Does it reach a *better* best?

### 🔧 Your turn 2

Set `max_depth=1` (stumps) and re-run the comparison against linear regression.

With depth-1 trees the model cannot represent interactions between predictors.
How much of boosting's advantage survives? That difference is how much of the
gain came from interactions rather than from curvature.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** With `learning_rate=0.3` the test error bottoms out much earlier
— a few dozen trees — but usually at a slightly worse minimum, and it degrades
faster afterwards. Learning rate and number of trees trade off against each other:
a small rate with many trees generally generalises better, at the cost of compute.
The pairing matters more than either number alone.

**Your turn 2.** Depth-1 stumps give an additive model — each predictor
contributes independently, with no interactions — and much of boosting's edge
over linear regression usually survives. That tells you the gain here comes mostly
from *curvature* (non-linear relationships between individual predictors and STIR)
rather than from interactions between them. That is a genuinely useful diagnostic:
if stumps do nearly as well, you can describe the model as "one curve per
variable", which is far easier to explain to a policy reader than a full
interaction model.

</details>

## Where this stops

You have a better predictor and no coefficients. The obvious question — *what
drives housing burden?* — is no longer answerable from the model's parameters.
The next notebook recovers an answer a different way.